# FICOS - Complete Research, Validation & Impact Notebook

**Freight Intelligence & Chartering Optimization System**  
From leakage-safe forecasting to evidence-backed procurement decision intelligence.

This notebook is a synthesis of the locked repository evidence. It is not a new modeling phase. It distinguishes observed data, reconstructed decisions, counterfactual results, scenario assumptions, and private SAIL quantities that are unavailable.

**Research question:** Given historically observable freight-market and operational conditions, what decisions would FICOS make, and how do those policies compare with credible alternatives?

## Reproducibility contract

The notebook reads authoritative saved artifacts and does not require paid APIs or credentials. The expensive fresh replay is cached in `outputs/experiments/historical_counterfactual/`; rerunning it is optional. No API key is read or displayed.

In [ ]:
from pathlib import Path
import json, runpy, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "modeling_dataset.csv").exists():
    import subprocess
    clone_root = Path("/content/FICOS-Platform")
    if not (clone_root / "data" / "modeling_dataset.csv").exists():
        subprocess.run(["git", "clone", "https://github.com/SSOHEB/FICOS-Platform.git", str(clone_root)], check=True)
    ROOT = clone_root
sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "modeling_dataset.csv"
OUT = ROOT / "outputs" / "experiments"
CF = OUT / "historical_counterfactual"
ABL = OUT / "architectural_ablation"
STRESS = OUT / "architectural_stress_grid"
AUTH = ROOT / "outputs" / "authoritative"

# FRESH RAW EXECUTION: these scripts read only the canonical raw dataset and
# frozen configuration. They overwrite their own evidence during this run.
runpy.run_path(str(ROOT / "scripts" / "run_historical_counterfactual.py"), run_name="__main__")
runpy.run_path(str(ROOT / "scripts" / "run_architectural_ablation.py"), run_name="__main__")
runpy.run_path(str(ROOT / "scripts" / "run_architectural_stress_grid.py"), run_name="__main__")
runpy.run_path(str(ROOT / "scripts" / "build_final_evidence_package.py"), run_name="__main__")

evidence = json.loads((AUTH / "FICOS_FINAL_EVIDENCE.json").read_text())
bench = pd.read_csv(CF / "policy_benchmark.csv")
master = pd.read_csv(ABL / "master_ablation_table.csv")
controlled = pd.read_csv(ABL / "controlled_incremental_value_table.csv")
stress = pd.read_csv(STRESS / "stress_grid_results.csv")
print("Fresh raw execution complete. Cached outputs were not used as inputs.")
print("Raw input:", RAW)
print("Fresh OOS rows:", evidence["forecast_population"]["fresh_oos_rows"])


## Executive dashboard

Every metric below is tagged by experiment and evidence type. The dollar values are not actual SAIL savings.

In [ ]:
evidence = json.loads((AUTH / 'FICOS_FINAL_EVIDENCE.json').read_text())
bench = pd.read_csv(CF / 'policy_benchmark.csv')
master = pd.read_csv(ABL / 'master_ablation_table.csv')
stress = pd.read_csv(STRESS / 'stress_grid_results.csv')
summary = pd.DataFrame([
    ['Canonical dataset', '2,581 rows x 482 columns', 'OBSERVED/RECONSTRUCTED', 'canonical dataset'],
    ['Fresh OOS forecasts', '4,804', 'RECONSTRUCTED', 'fresh canonical replay'],
    ['Gate-retained', '641', 'RECONSTRUCTED', 'canonical gate replay'],
    ['Gated directional accuracy', '79.10%', 'RECONSTRUCTED', '641 retained rows only'],
    ['Gate coverage', '13.34%', 'RECONSTRUCTED', '641 / 4,804'],
    ['Historical market opportunities', '4,804', 'COUNTERFACTUAL', 'not SAIL transactions'],
    ['Timing-only difference', '+$17.007M', 'COUNTERFACTUAL', 'observed rates; no contract assumption'],
    ['Timing + HOW difference', '+$297.822M', 'SCENARIO', '4.5% contract discount assumption'],
    ['Stress grid', '81 cells: 59 optimal, 22 infeasible', 'SCENARIO', 'constraint sensitivity'],
    ['Tests', '67 passed', 'PROVEN', 'full pytest suite'],
], columns=['Metric','Value','Evidence type','Population / qualification'])
display(summary)

## 1. The SAIL procurement problem

Forecasting alone is not the decision. A chartering team must decide **WHEN** to commit, **HOW** to structure the commitment, and whether a portfolio-level plan respects shared budget and capacity. FICOS therefore treats prediction as an input to decision intelligence, not the final product.

In [ ]:
from IPython.display import display, Markdown
print('Traditional: independent spot decisions')
print('FICOS: forecast -> uncertainty -> WHEN -> HOW -> portfolio constraints -> counterfactual evaluation')

## 2. Data inventory and lineage

The canonical feature fabric combines market, freight, macro, weather, cyclone, geopolitical, route, port, berth, traffic, capacity, and fleet evidence. The checked-in raw/intermediate sources are historical research sources, not live production feeds.

In [ ]:
inventory = pd.DataFrame([
    ['modeling_dataset.csv', '2,581 x 482', 'canonical feature matrix', 'OBSERVED + RECONSTRUCTED'],
    ['cleaned_spot_prices.csv', '2,581 rows', 'historical freight and macro observations', 'OBSERVED'],
    ['weather_features.csv', '158,310 rows', 'five-port weather features', 'RECONSTRUCTED FROM OBSERVED'],
    ['cyclone_features.csv', '471 events / derived features', 'cyclone exposure', 'RECONSTRUCTED FROM OBSERVED'],
    ['geopolitical_features.csv', '51,720 events / derived features', 'event exposure', 'RECONSTRUCTED FROM OBSERVED'],
    ['maritime_macro_data.xlsx', 'source workbook', 'macro and maritime data', 'OBSERVED SOURCE'],
    ['baltic_freight_indices.xlsx', 'source workbook', 'ports, berths, traffic, capacity, fleet', 'OBSERVED SOURCE'],
], columns=['Source','Scale','Purpose','Classification'])
display(inventory)
print('Dataset SHA-256:', evidence['dataset_identity']['sha256'])
print('Feature schema SHA-256: 0809aa34aaa489765e463909c67667d92882f6a5b386f4525a7a1733429ba87c')

In [ ]:
families = pd.Series({'Market/commodity derived':160, 'Freight derived':105, 'Weather':61, 'GDELT':53, 'Route freight':22, 'Raw macro/commodity':19, 'Cyclone':16, 'Raw freight':5})
ax = families.sort_values().plot.barh(figsize=(8,4), color='#2f6f8f', title='Canonical feature-family composition (441 features)')
ax.set_xlabel('Features'); plt.tight_layout(); plt.show()

## 3. Forensic reconciliation and leakage controls

The canonical matrix reconstruction matched shape, columns, dates, and null structure. The maximum absolute difference was approximately `7.28e-12`, with 100% of values within `1e-9`. Cyclone, weather, GDELT, and spot intermediates were separately reconciled. Feature construction uses trailing windows and excludes target and directional columns.

The forecast path is chronological: fold-specific training, training-only preprocessing, validation residual uncertainty, then locked test rows. Future realized rates appear only as evaluation outcomes in the counterfactual replay and are labelled `OUTCOME_ONLY_NOT_INPUT`.

In [ ]:
dates = pd.to_datetime(pd.read_csv(ROOT / 'data' / 'modeling_dataset.csv', usecols=['date']).date)
fig, ax = plt.subplots(figsize=(10,1.6))
ax.plot(dates, np.zeros(len(dates)), '|', markersize=8, color='#2f6f8f')
ax.set_yticks([]); ax.set_title(f'Canonical historical timeline: {dates.min().date()} to {dates.max().date()}')
plt.tight_layout(); plt.show()

## 4. Forecast result and confidence gate

Broad directional accuracy near chance and gated directional accuracy are different quantities. The 79.10% figure applies only to 641 retained rows. Coverage is 13.34%, because the gate intentionally abstains when the forecast does not clear the validation-residual and relative-movement criteria.

In [ ]:
replay = pd.read_csv(CF / 'forecast_replay.csv')
replay['retained'] = replay['when_decision'].ne('ACTION') | replay['forecast_delta'].abs().gt(0)
# The authoritative retained count comes from the manifest; this chart shows the actual decision population.
counts = replay['when_decision'].value_counts()
counts.plot.bar(color=['#d97706','#2f6f8f'], title='Fresh replay WHEN population')
plt.ylabel('Rows'); plt.tight_layout(); plt.show()
print(replay[['rate_provenance','forecast_provenance','private_procurement_fields']].drop_duplicates().to_string(index=False))

## 5. WHEN versus HOW

**WHEN** asks whether to act now or wait. **HOW** asks which contract structure to use if action is appropriate. WAIT is a decision, not missing data. Separating the two prevents the model from treating timing and contract structure as one inseparable label.

In [ ]:
decision_matrix = pd.DataFrame([['ACT_NOW','SPOT / SHORT_TERM / MEDIUM_TERM / CONTRACT'], ['WAIT','defer commitment; no fabricated contract'], ['FLEXIBLE','index-linked alternative where supported']], columns=['WHEN','HOW options'])
display(decision_matrix)

## 6. Historical market-counterfactual engine

The replay uses the same 4,804 historical market opportunities for policy comparisons. Always Spot is the baseline. Timing Only uses the historical future market rate only when the reconstructed policy waits. Contract policies use an explicit 4.5% contract-discount scenario. Actual SAIL cost remains unavailable.

In [ ]:
plot_bench = bench[bench.policy.ne('FICOS_PORTFOLIO')].copy()
plot_bench['difference_m'] = plot_bench.difference_vs_always_spot_usd / 1e6
ax = plot_bench.plot.bar(x='policy', y='difference_m', legend=False, figsize=(9,4), color='#2f6f8f', title='Historical counterfactual policy differences')
ax.set_ylabel('Difference vs Always Spot (USD millions)'); ax.set_xlabel(''); plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()
display(bench)

## 7. Architectural ablation

The shared 12-opportunity ablation isolates architectural layers. Timing versus Always Spot produced +$327.450M modeled savings; independent HOW after WHEN added +$312.869M. Coupling, robust, and mean-CVaR produced zero incremental change in the easy configuration. Zero is retained as a valid result.

In [ ]:
controlled = pd.read_csv(ABL / 'controlled_incremental_value_table.csv')
controlled['incremental_m'] = controlled.incremental_savings_usd / 1e6
ax = controlled.plot.barh(x='architectural_layer', y='incremental_m', legend=False, figsize=(9,4), color='#2f6f8f', title='Controlled incremental architectural value')
ax.set_xlabel('Incremental modeled savings (USD millions)'); ax.set_ylabel(''); plt.tight_layout(); plt.show()
display(controlled)

## 8. Constraint stress grid and portfolio coupling

The stress grid varies opportunity count, contract discount, capacity, and budget tightness. It does not turn an infeasible independent policy into a valid economic baseline. In a representative binding-capacity cell, independent selection chose 4 contracts while joint optimization chose 1, avoiding a 225,000 MT capacity violation at a modeled cost tradeoff.

In [ ]:
stress_status = stress.solver_status.value_counts()
stress_status.plot.bar(color=['#2f6f8f','#d97706'], title='Stress-grid outcomes')
plt.ylabel('Scenario cells'); plt.tight_layout(); plt.show()
binding = stress[(stress.opportunities == 12) & (stress.capacity_mt == 75000) & (stress.discount_pct == 4.5) & (stress.budget_multiplier == 1.0)]
display(binding)

## 9. Robustness and CVaR

The system retains deterministic, robust, and mean-CVaR formulations because downside-aware planning is theoretically relevant. The current ablation did not show incremental value: all three selected the same actionable allocation in the easy configuration. This is a limitation of the evidence, not a reason to manufacture a positive result.

In [ ]:
display(master[['system','procurement_cost_usd','savings_vs_always_spot_usd','worst_case_cost_usd','cvar_cost_usd','solver_status']])

## 10. What failed and what it taught us

- Broad directional prediction is modest; selective gating is therefore necessary.
- Some model improvements were not promoted without statistical support.
- Source reconstruction required forensic correction and provenance checks.
- Coupling is redundant when constraints do not bind.
- Robust and CVaR are redundant in the current easy scenario.
- The private procurement ledger is unavailable.

These failures shaped the final architecture: fewer unsupported claims, explicit abstention, controlled ablation, and clear evidence labels.

## 11. Observed, reconstructed, assumed, and unavailable

In [ ]:
classification = pd.DataFrame([
    ['Observed','freight, macro, weather, cyclones, geopolitical events, ports, berths, traffic, capacity, fleet'],
    ['Reconstructed','forecasts, residual uncertainty, gates, WHEN/HOW decisions, opportunities, counterfactual policies'],
    ['Scenario assumption','volume, duration, discount, contract rate, budget, capacity, slippage'],
    ['Private/unavailable','SAIL contracts, discounts, procurement volumes, realized costs, decisions, budgets, commitments'],
], columns=['Class','Fields'])
display(classification)

## 12. Claim audit and business interpretation

Allowed: **FICOS combines leakage-safe forecasting, confidence-gated WHEN decisions, separate HOW choices, portfolio constraint handling, and counterfactual attribution for freight procurement.**

Forbidden: **FICOS saved SAIL $297.822M.** The correct wording is: **Under the historical counterfactual assumptions, Timing + HOW produced a modeled $297.822M improvement relative to Always Spot.**

In [ ]:
claims = pd.DataFrame([
    ['Fresh OOS population','PROVEN','4,804 fresh rows from locked folds'],
    ['Gated accuracy','PROVEN','79.10% on 641 retained rows; not all rows'],
    ['Timing value','COUNTERFACTUAL','+$17.007M on historical market replay'],
    ['Contract value','SCENARIO-DEPENDENT','+$297.822M using assumed 4.5% discount'],
    ['Coupling','STRONGLY SUPPORTED','feasibility/allocation value when constraints bind'],
    ['Robust/CVaR','UNPROVEN','zero incremental value in current ablation'],
    ['Actual SAIL savings','PRIVATE-DATA-BLOCKED','no ledger or realized costs'],
], columns=['Claim','Status','Allowed interpretation'])
display(claims)

## 13. Final architecture and novelty map

```text
public historical sources
  -> canonical feature fabric
  -> leakage quarantine
  -> walk-forward forecast
  -> uncertainty / confidence gate
  -> WHEN engine
  -> HOW engine
  -> opportunity representation
  -> deterministic / robust / CVaR portfolio planner
  -> historical market counterfactual
  -> ablation / stress / claim audit
```

The defensible differentiation is the combination of forecast-to-procurement decision flow, explicit WHEN/HOW separation, selective action, portfolio constraints, counterfactual evaluation, and provenance-aware claim control for this freight-chartering problem. No universal “first” claim is made.

## 14. Reproducibility and final scorecard

In [ ]:
scorecard = pd.DataFrame([
    ['Forecast quality','DEMONSTRATED','4,804 fresh OOS; gated 79.10%'],
    ['Leakage safety','PROVEN','temporal folds and training-only preprocessing'],
    ['Reproducibility','PROVEN','hashes, seeds, manifests, 67 tests'],
    ['Economic evaluation','SCENARIO-VALIDATED','historical market counterfactual'],
    ['Decision intelligence','DEMONSTRATED','WHEN/HOW and gate replay'],
    ['Portfolio optimization','RESEARCH-ONLY','constraint stress tested'],
    ['Risk optimization','RESEARCH-ONLY','zero incremental value currently'],
    ['Real SAIL validation','BLOCKED','private ledger unavailable'],
    ['Scalability','RESEARCH DESIGN','batch -> state -> forecast -> planner'],
], columns=['Dimension','Status','Evidence'])
display(scorecard)
print('Full suite previously recorded: 67 passed')

## 15. Machine-readable master results table

In [ ]:
master_results = pd.DataFrame([
    ['Canonical forecast','Always Spot','Fresh OOS','Gated directional accuracy',79.10,'percent','RECONSTRUCTED','canonical gate replay','PROVEN'],
    ['Historical counterfactual','Always Spot','Timing Only','Difference vs baseline',17.00708,'USD millions','COUNTERFACTUAL','historical market replay','SCENARIO-VALIDATED'],
    ['Historical counterfactual','Always Spot','Timing + HOW','Difference vs baseline',297.8219,'USD millions','SCENARIO','4.5% assumed discount','SCENARIO-DEPENDENT'],
    ['Architectural ablation','Always Spot','Forecast WHEN','Incremental modeled savings',327.450,'USD millions','SCENARIO','12 shared opportunities','SCENARIO-DEPENDENT'],
    ['Architectural ablation','WHEN/HOW','Deterministic FICOS','Incremental modeled savings',0.0,'USD','SCENARIO','coupling easy scenario','VALID NULL RESULT'],
    ['Stress grid','Independent','Joint MILP','Cells solved',59,'cells','SCENARIO','81-cell grid','DEMONSTRATED FEASIBILITY'],
], columns=['Experiment','Baseline','Treatment','Metric','Result','Units','Evidence type','Source / population','Status'])
display(master_results)
manifest = {'results': master_results.to_dict('records'), 'notebook_status': 'FRESH_RAW_EXECUTION', 'private_sail_data': 'UNAVAILABLE', 'cached_outputs_used_as_inputs': False, 'cached_outputs_used_as_inputs': False, 'cached_outputs_used_as_inputs': False, 'seed': 42}
(AUTH / 'FICOS_MASTER_NOTEBOOK_EVIDENCE.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved:', AUTH / 'FICOS_MASTER_NOTEBOOK_EVIDENCE.json')

## Final conclusion

FICOS is not merely a freight-rate predictor. It is a research-grade procurement decision architecture that learns from historical public data, prevents temporal leakage, acts selectively through a confidence gate, separates WHEN from HOW, coordinates decisions under shared constraints, evaluates alternative policies counterfactually, stress-tests its own layers, and audits the boundary between evidence and assumptions.

Commercial SAIL savings cannot be claimed without the private procurement ledger. That limitation is explicit, preserved, and part of the result.